# Math Client — standalone full-parameter OPSD

This notebook implements OPSD directly with PyTorch and Transformers. It does not clone, import,
patch, or execute the authors' repository.

- Student sees only the problem and samples one on-policy completion.
- A frozen step-0 Teacher sees the problem plus the privileged reference solution.
- Both models score the exact same sampled completion at aligned completion-token positions.
- The objective is full-vocabulary forward KL, `KL(Teacher || Student)`, with temperature 1.1.
- Each vocabulary contribution is upper-clipped at 0.05 before reduction.
- Only the full-parameter Student receives gradients; LoRA and quantization are not used.
- Training runs for 100 optimizer steps with effective batch 32 on one RTX 4090 48 GB.
- A resumable checkpoint is saved every 10 steps and only the newest checkpoint is retained.

In [ ]:
# Papermill parameters
GPU_ID = "0"
MODEL_NAME = "Qwen/Qwen3-4B"
MODEL_SOURCE = "Qwen3-4B"
MODEL_REVISION = "main"
MODEL_LOCAL_FILES_ONLY = True
DATASET_NAME = "siyanzhao/Openthoughts_math_30k_opsd"
DATASET_REVISION = "1f33e9dc2e8a1c639ca74f8024ad4a9f1f5eae62"
DATASET_LOCAL_DIR = "Openthoughts_math_30k_opsd"
TRAIN_LOCAL_FILES_ONLY = True

OUTPUT_ROOT = "results/client_math_opsd_qwen3_4b_fullft"
RUN_NAME = "standalone_math_fullft_100steps_single_4090"
SEED = 42
MAX_STEPS = 100
MICRO_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 32
EFFECTIVE_BATCH_SIZE = 32
ROLLOUTS_PER_PROMPT = 1
LEARNING_RATE = 5e-6
WEIGHT_DECAY = 0.0
MAX_GRAD_NORM = 0.1
WARMUP_RATIO = 0.0
LR_SCHEDULER = "linear"
MAX_COMPLETION_LENGTH = 1024
MAX_LENGTH = 20000
TRAIN_TEMPERATURE = 1.1
TRAIN_TOP_P = 0.95
TRAIN_TOP_K = 20
DISTILL_TEMPERATURE = 1.1
BETA = 0.0
POINTWISE_KL_CLIP = 0.05
SAVE_STEPS = 10
SAVE_TOTAL_LIMIT = 1
ATTN_IMPLEMENTATION = "sdpa"
SKIP_TRAINING_IF_COMPLETE = True

# Matched AIME evaluation protocol.
LOCAL_FILES_ONLY = True
LIMIT_AIME25 = None
LIMIT_AIME26 = None
EVAL_MAX_SEQ_LENGTH = 32768
EVAL_MAX_NEW_TOKENS = 1024
EVAL_TEMPERATURE = 0.6
EVAL_TOP_P = 0.95
EVAL_TOP_K = 20
EVAL_MIN_P = 0.0
EVAL_ENABLE_THINKING = False
EVAL_SAMPLES_PER_PROMPT = 12
RESUME_EVALUATION = True
MAX_CONSECUTIVE_EVAL_ERRORS = 3

## Runtime

The existing `py312` environment is used. No paper package versions are pinned. PyTorch SDPA is
used, so the external `flash_attn` package is not required. The source checkpoint is the
post-trained Qwen3-4B checkpoint, not Qwen3-4B-Base.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU_ID)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("WANDB_MODE", "disabled")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc, hashlib, importlib.metadata, json, random, shutil, time
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from datasets import DownloadConfig, disable_progress_bars, load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, get_scheduler

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Start the RTX 4090 instance and restart the kernel.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("The visible GPU does not support BF16.")
if MICRO_BATCH_SIZE != 1 or EFFECTIVE_BATCH_SIZE != GRADIENT_ACCUMULATION_STEPS:
    raise ValueError("This implementation requires micro-batch 1 and effective batch 32.")
if ROLLOUTS_PER_PROMPT != 1 or BETA != 0.0:
    raise ValueError("This run requires one rollout and forward KL (beta=0).")

device = torch.device("cuda:0")
torch.set_float32_matmul_precision("high")
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
workspace = Path.cwd().resolve()
output_root = (workspace / OUTPUT_ROOT).resolve()
run_dir = output_root / RUN_NAME
run_dir.mkdir(parents=True, exist_ok=True)
local_model = Path(MODEL_SOURCE).expanduser()
model_ref = str(local_model.resolve()) if (local_model / "config.json").is_file() else MODEL_NAME

def source_kwargs(source, local_only):
    kwargs = {"local_files_only": bool(local_only), "trust_remote_code": False}
    if not Path(str(source)).is_dir():
        kwargs["revision"] = MODEL_REVISION
    return kwargs

packages = ["torch", "transformers", "datasets", "accelerate", "safetensors", "tokenizers"]
installed = {}
for package in packages:
    try: installed[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError: installed[package] = None
print("Packages:", json.dumps(installed, ensure_ascii=False))
print(f"GPU: {torch.cuda.get_device_name(0)}; VRAM={torch.cuda.get_device_properties(0).total_memory/2**30:.2f} GiB")
print(f"Model: {model_ref}\nOutput: {run_dir}")

## Standalone OPSD objective

For Student rollout $\hat y\sim p_\theta(\cdot\mid x)$, Student and frozen Teacher distributions are
$p_S^t=p_\theta(\cdot\mid x,\hat y_{<t})$ and
$p_T^t=p_{\theta_0}(\cdot\mid x,y^*,\hat y_{<t})$. The loss is

$$\mathcal L=\frac1{|\hat y|}\sum_{t,v}\min\{p_T^t(v)[\log p_T^t(v)-\log p_S^t(v)],0.05\}.$$

The prompts may have different lengths, but logits are sliced from the first completion token, so
the two distributions are aligned by completion position.

In [ ]:
disable_progress_bars()
local_dataset_dir = Path(DATASET_LOCAL_DIR).expanduser().resolve()
local_train_shards = sorted(local_dataset_dir.glob("train-*-of-*.parquet"))
if local_train_shards:
    print(f"Loading {len(local_train_shards)} local training shards from {local_dataset_dir}", flush=True)
    train_dataset = load_dataset(
        "parquet", data_files={"train": [str(path) for path in local_train_shards]}, split="train"
    )
elif TRAIN_LOCAL_FILES_ONLY:
    raise FileNotFoundError(
        f"No local train-*.parquet shards found in {local_dataset_dir}. "
        "Run download_math_30k_opsd.ipynb from the same working directory first."
    )
else:
    train_dataset = load_dataset(DATASET_NAME, revision=DATASET_REVISION, split="train")
needed = MAX_STEPS * GRADIENT_ACCUMULATION_STEPS
if len(train_dataset) < needed:
    raise ValueError(f"Need {needed} rows, found {len(train_dataset)}")
g = torch.Generator(device="cpu"); g.manual_seed(SEED)
sample_order = torch.randperm(len(train_dataset), generator=g).tolist()

tokenizer = AutoTokenizer.from_pretrained(model_ref, **source_kwargs(model_ref, MODEL_LOCAL_FILES_ONLY))
if tokenizer.pad_token_id is None: tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"; tokenizer.truncation_side = "right"
TRANSITION = (
    "\n\nAfter reading the reference solution above, understand every step rather than copying it. "
    "Now independently derive the same final answer. Think step by step and put the final answer within \\boxed{}."
)

def render_student(problem):
    content = f"Problem: {str(problem).strip()}\n\nPlease reason step by step, and put your final answer within \\boxed{{}}."
    return tokenizer.apply_chat_template([{"role":"user","content":content}], tokenize=False,
        add_generation_prompt=True, enable_thinking=False)

def render_teacher(problem, solution):
    content = (f"Problem: {str(problem).strip()}\n\nHere is a correct reference solution:\n"
        f"=== Reference Solution Begin ===\n{str(solution).strip()}\n=== Reference Solution End ==={TRANSITION}")
    return tokenizer.apply_chat_template([{"role":"user","content":content}], tokenize=False,
        add_generation_prompt=True, enable_thinking=True)

def encode_prompt(text):
    batch = tokenizer(text, return_tensors="pt", add_special_tokens=False, truncation=True,
        max_length=MAX_LENGTH-MAX_COMPLETION_LENGTH)
    return batch["input_ids"].to(device), batch["attention_mask"].to(device)

print(f"Training rows={len(train_dataset)}; rows consumed={needed}")

## Checkpoint and resume

Each checkpoint contains Student weights, tokenizer, optimizer, scheduler, RNG, data cursor, and
loss history. A new checkpoint is completed before the preceding checkpoint is deleted.

In [ ]:
def write_json(path, payload):
    path = Path(path); temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    temp.replace(path)

def checkpoint_step(path): return int(path.name.rsplit("-", 1)[-1])
def checkpoints(): return sorted([p for p in run_dir.glob("checkpoint-*") if p.is_dir()], key=checkpoint_step)
def latest_checkpoint(): return checkpoints()[-1] if checkpoints() else None

def training_state():
    path = run_dir / "training_state.json"
    return json.loads(path.read_text(encoding="utf-8")) if path.is_file() else None

def training_complete():
    state = training_state()
    return state is not None and int(state.get("global_step", -1)) >= MAX_STEPS and (run_dir/"config.json").is_file()

def rng_state():
    return {"python":random.getstate(), "numpy":np.random.get_state(),
        "torch_cpu":torch.get_rng_state(), "torch_cuda":torch.cuda.get_rng_state_all()}

def restore_rng(state):
    random.setstate(state["python"]); np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch_cpu"]); torch.cuda.set_rng_state_all(state["torch_cuda"])

def optimizer_to(optimizer, target):
    for state in optimizer.state.values():
        for key, value in state.items():
            if torch.is_tensor(value): state[key] = value.to(target)

def save_checkpoint(student, optimizer, scheduler, state):
    step = int(state["global_step"]); target = run_dir/f"checkpoint-{step:03d}"; temp = run_dir/f".checkpoint-{step:03d}.tmp"
    if temp.exists(): shutil.rmtree(temp)
    temp.mkdir(parents=True)
    student.save_pretrained(temp, safe_serialization=True, max_shard_size="2GB")
    tokenizer.save_pretrained(temp)
    torch.save(optimizer.state_dict(), temp/"optimizer.pt")
    torch.save(scheduler.state_dict(), temp/"scheduler.pt")
    torch.save(rng_state(), temp/"rng_state.pt")
    write_json(temp/"training_state.json", state)
    temp.rename(target); write_json(run_dir/"training_state.json", state)
    for old in [p for p in checkpoints() if p != target]: shutil.rmtree(old)
    print(f"[CHECKPOINT] {target} (latest only)", flush=True)

## On-policy rollout and forward KL

The Teacher does not generate a replacement response. It scores the Student's exact rollout while
conditioned on the privileged reference solution; gradients exist only through Student logits.

In [ ]:
def opsd_loss(student, teacher, example):
    s_prompt, s_mask = encode_prompt(render_student(example["problem"])); s_len = s_prompt.shape[1]
    student.eval()
    with torch.inference_mode():
        generated = student.generate(input_ids=s_prompt, attention_mask=s_mask,
            max_new_tokens=MAX_COMPLETION_LENGTH, do_sample=True, temperature=TRAIN_TEMPERATURE,
            top_p=TRAIN_TOP_P, top_k=TRAIN_TOP_K, eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id, use_cache=True)
    student.train()
    completion = generated[:, s_len:].detach(); n = completion.shape[1]
    if n == 0: raise RuntimeError("Student generated zero tokens")
    t_prompt, _ = encode_prompt(render_teacher(example["problem"], example["solution"])); t_len = t_prompt.shape[1]
    s_input = torch.cat([s_prompt, completion], 1); t_input = torch.cat([t_prompt, completion], 1)

    with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
        t_out = teacher(input_ids=t_input, attention_mask=torch.ones_like(t_input), use_cache=False)
        t_logits = t_out.logits[:, t_len-1:t_len-1+n, :]
        t_logp = F.log_softmax(t_logits / DISTILL_TEMPERATURE, dim=-1).detach()
    del t_out, t_logits, t_input

    with torch.autocast("cuda", dtype=torch.bfloat16):
        s_out = student(input_ids=s_input, attention_mask=torch.ones_like(s_input), use_cache=False)
        s_logits = s_out.logits[:, s_len-1:s_len-1+n, :]
        s_logp = F.log_softmax(s_logits / DISTILL_TEMPERATURE, dim=-1)
        contributions = F.kl_div(s_logp, t_logp, reduction="none", log_target=True)
        loss = contributions.clamp(max=POINTWISE_KL_CLIP).sum() / n
    del s_out, s_logits, s_logp, t_logp, contributions, s_input, s_prompt, s_mask, t_prompt, completion, generated
    return loss, int(n)

## Train for 100 optimizer steps

In [ ]:
config = {"implementation":"standalone PyTorch/Transformers", "external_opsd_code":False,
    "model":MODEL_NAME, "model_source":model_ref, "dataset":DATASET_NAME,
    "dataset_source":str(local_dataset_dir), "dataset_revision":DATASET_REVISION,
    "objective":"full-vocabulary KL(teacher || student)", "pointwise_clip":POINTWISE_KL_CLIP,
    "student_thinking":False, "teacher_thinking":True, "fixed_teacher":True,
    "full_parameter":True, "learning_rate":LEARNING_RATE, "effective_batch":EFFECTIVE_BATCH_SIZE,
    "max_steps":MAX_STEPS, "max_completion_length":MAX_COMPLETION_LENGTH,
    "packages":installed, "created_at_utc":datetime.now(timezone.utc).isoformat()}
write_json(output_root/"experiment_config.json", config)

if SKIP_TRAINING_IF_COMPLETE and training_complete():
    print(f"[SKIP] Complete Student already exists: {run_dir}")
else:
    resume = latest_checkpoint(); student_source = resume if resume else model_ref
    print(f"Loading Student: {student_source}", flush=True)
    student = AutoModelForCausalLM.from_pretrained(student_source, dtype=torch.bfloat16,
        attn_implementation=ATTN_IMPLEMENTATION, **source_kwargs(student_source, MODEL_LOCAL_FILES_ONLY)).to(device)
    student.config.use_cache = False
    try: student.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant":False})
    except TypeError: student.gradient_checkpointing_enable()
    student.train()
    print(f"Loading frozen Teacher: {model_ref}", flush=True)
    teacher = AutoModelForCausalLM.from_pretrained(model_ref, dtype=torch.bfloat16,
        attn_implementation=ATTN_IMPLEMENTATION, **source_kwargs(model_ref, MODEL_LOCAL_FILES_ONLY)).to(device)
    teacher.requires_grad_(False); teacher.eval(); teacher.config.use_cache = False
    optimizer = torch.optim.AdamW(student.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, fused=True)
    scheduler = get_scheduler(LR_SCHEDULER, optimizer=optimizer,
        num_warmup_steps=int(MAX_STEPS*WARMUP_RATIO), num_training_steps=MAX_STEPS)

    if resume:
        state = json.loads((resume/"training_state.json").read_text(encoding="utf-8"))
        optimizer.load_state_dict(torch.load(resume/"optimizer.pt", map_location="cpu", weights_only=False)); optimizer_to(optimizer, device)
        scheduler.load_state_dict(torch.load(resume/"scheduler.pt", map_location="cpu", weights_only=False))
        restore_rng(torch.load(resume/"rng_state.pt", map_location="cpu", weights_only=False))
        start = int(state["global_step"]); cursor = int(state["data_cursor"]); history = list(state["loss_history"])
        print(f"[RESUME] step={start} from {resume}")
    else: start, cursor, history = 0, 0, []

    def format_elapsed(seconds):
        seconds = max(0, int(seconds))
        hours, remainder = divmod(seconds, 3600)
        minutes, seconds = divmod(remainder, 60)
        return f"{hours:02d}:{minutes:02d}:{seconds:02d}"

    torch.cuda.reset_peak_memory_stats(0); started = time.time()
    for step_index in range(start, MAX_STEPS):
        optimizer.zero_grad(set_to_none=True); total_loss = 0.0; total_tokens = 0
        for micro_index in range(GRADIENT_ACCUMULATION_STEPS):
            example = train_dataset[sample_order[cursor % len(sample_order)]]; cursor += 1
            loss, tokens = opsd_loss(student, teacher, example)
            sample_loss = float(loss.detach().cpu())
            (loss/GRADIENT_ACCUMULATION_STEPS).backward()
            total_loss += sample_loss; total_tokens += tokens
            completed_samples = step_index * GRADIENT_ACCUMULATION_STEPS + micro_index + 1
            progress = 100.0 * completed_samples / needed
            print(
                f"[PROGRESS] train/math: {completed_samples}/{needed} ({progress:.2f}%) "
                f"loss={sample_loss:.6f} rollouts={ROLLOUTS_PER_PROMPT} "
                f"mean_tokens={float(tokens):.1f} opt_step={step_index}/{MAX_STEPS} "
                f"elapsed={format_elapsed(time.time()-started)}",
                flush=True,
            )
            del loss; torch.cuda.empty_cache()
        grad_norm = torch.nn.utils.clip_grad_norm_(student.parameters(), MAX_GRAD_NORM)
        optimizer.step(); scheduler.step(); optimizer.zero_grad(set_to_none=True)
        step = step_index + 1
        row = {"step":step, "loss":total_loss/GRADIENT_ACCUMULATION_STEPS,
            "learning_rate":float(scheduler.get_last_lr()[0]), "grad_norm":float(grad_norm.detach().cpu()),
            "completion_tokens":total_tokens, "data_cursor":cursor, "elapsed_seconds":time.time()-started}
        history.append(row)
        print(
            f"[STEP LOSS] train/math: step={step}/{MAX_STEPS} loss={row['loss']:.6f} "
            f"samples={GRADIENT_ACCUMULATION_STEPS} grad={row['grad_norm']:.5f} "
            f"lr={row['learning_rate']:.3e} mean_tokens={total_tokens/GRADIENT_ACCUMULATION_STEPS:.1f} "
            f"peak={torch.cuda.max_memory_allocated()/2**30:.2f} GiB",
            flush=True,
        )
        state = {"status":"training", "global_step":step, "data_cursor":cursor,
            "loss_history":history, "updated_at_utc":datetime.now(timezone.utc).isoformat()}
        if step % SAVE_STEPS == 0 or step == MAX_STEPS: save_checkpoint(student, optimizer, scheduler, state)

    student.save_pretrained(run_dir, safe_serialization=True, max_shard_size="2GB"); tokenizer.save_pretrained(run_dir)
    write_json(run_dir/"training_state.json", {"status":"complete", "global_step":MAX_STEPS,
        "data_cursor":cursor, "loss_history":history, "completed_at_utc":datetime.now(timezone.utc).isoformat()})
    print(f"[COMPLETE] {run_dir}")
    del optimizer, scheduler, teacher, student; gc.collect(); torch.cuda.empty_cache()

## Save losses and training curve

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
state = training_state()
if state is None or int(state.get("global_step", -1)) < MAX_STEPS: raise RuntimeError("Training is incomplete")
losses = pd.DataFrame(state["loss_history"]).drop_duplicates("step", keep="last").sort_values("step")
loss_csv = output_root/"optimizer_step_losses.csv"; loss_json = output_root/"optimizer_step_losses.json"; loss_png = output_root/"optimizer_step_loss_curve.png"
losses.to_csv(loss_csv, index=False); loss_json.write_text(losses.to_json(orient="records", indent=2), encoding="utf-8")
fig, ax = plt.subplots(figsize=(10,5.5)); ax.plot(losses.step, losses.loss, alpha=.55, label="step loss")
ax.plot(losses.step, losses.loss.rolling(10,min_periods=1).mean(), linewidth=2.2, label="10-step moving average")
ax.set(title="Standalone Math Client OPSD training loss", xlabel="Optimizer step", ylabel="Forward-KL loss")
ax.grid(alpha=.2); ax.legend(); fig.tight_layout(); fig.savefig(loss_png, dpi=180); plt.show()
manifest = {"status":"complete", "implementation":"standalone", "global_step":int(state["global_step"]),
    "final_student":str(run_dir), "latest_checkpoint":str(latest_checkpoint()),
    "loss_csv":str(loss_csv), "loss_json":str(loss_json), "loss_curve":str(loss_png)}
write_json(output_root/"training_manifest.json", manifest); display(losses.tail(10)); print(json.dumps(manifest, indent=2))

## Post-training evaluation — AIME25 and AIME26

In [ ]:
import gc
import random
import re

import numpy as np
import pandas as pd
from datasets import DownloadConfig, disable_progress_bars, load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(0)

evaluation_dir = output_root / "evaluation"
raw_evaluation_dir = evaluation_dir / "raw"
raw_evaluation_dir.mkdir(parents=True, exist_ok=True)

if not run_dir.is_dir():
    raise FileNotFoundError(f"Final Student directory not found: {run_dir}")

print(f"Loading final full Student for evaluation: {run_dir}", flush=True)
eval_tokenizer_source = run_dir if (run_dir / "tokenizer_config.json").is_file() else model_ref
eval_tokenizer = AutoTokenizer.from_pretrained(
    eval_tokenizer_source,
    revision=None if Path(str(eval_tokenizer_source)).is_dir() else MODEL_REVISION,
    local_files_only=bool(LOCAL_FILES_ONLY),
    trust_remote_code=True,
)
eval_model = AutoModelForCausalLM.from_pretrained(
    run_dir,
    dtype=torch.bfloat16,
    attn_implementation=ATTN_IMPLEMENTATION,
    local_files_only=True,
    trust_remote_code=True,
).to("cuda:0")
eval_model.eval()
eval_model.config.use_cache = True
eval_model.generation_config.max_length = None

if eval_tokenizer.pad_token_id is None:
    eval_tokenizer.pad_token = eval_tokenizer.eos_token
eval_model.config.pad_token_id = eval_tokenizer.pad_token_id
eval_model.generation_config.pad_token_id = eval_tokenizer.pad_token_id

local_download_config = DownloadConfig(local_files_only=bool(LOCAL_FILES_ONLY))
disable_progress_bars()
aime_datasets = {
    "aime25": load_dataset(
        "math-ai/aime25", split="test", download_config=local_download_config
    ),
    "aime26": load_dataset(
        "math-ai/aime26", split="test", download_config=local_download_config
    ),
}
if LIMIT_AIME25 is not None:
    aime_datasets["aime25"] = aime_datasets["aime25"].select(
        range(min(int(LIMIT_AIME25), len(aime_datasets["aime25"])))
    )
if LIMIT_AIME26 is not None:
    aime_datasets["aime26"] = aime_datasets["aime26"].select(
        range(min(int(LIMIT_AIME26), len(aime_datasets["aime26"])))
    )
for benchmark, dataset in aime_datasets.items():
    required_columns = {"problem", "answer"}
    if not required_columns.issubset(dataset.column_names):
        raise ValueError(f"{benchmark} columns are incompatible: {dataset.column_names}")

eval_fingerprint_payload = {
    "model_dir": str(run_dir),
    "global_step": MAX_STEPS,
    "protocol": "project-math-avg12-v1",
    "max_seq_length": EVAL_MAX_SEQ_LENGTH,
    "max_new_tokens": EVAL_MAX_NEW_TOKENS,
    "temperature": EVAL_TEMPERATURE,
    "top_p": EVAL_TOP_P,
    "top_k": EVAL_TOP_K,
    "min_p": EVAL_MIN_P,
    "enable_thinking": EVAL_ENABLE_THINKING,
    "samples_per_prompt": EVAL_SAMPLES_PER_PROMPT,
    "seed": SEED,
}
EVAL_RUN_FINGERPRINT = hashlib.sha256(
    json.dumps(eval_fingerprint_payload, sort_keys=True).encode("utf-8")
).hexdigest()[:16]
print(f"Evaluation fingerprint: {EVAL_RUN_FINGERPRINT}", flush=True)
print(
    f"Evaluation CUDA allocation: {torch.cuda.memory_allocated(0) / 2**30:.2f} GiB",
    flush=True,
)


In [ ]:
def eval_format_seconds(seconds):
    seconds = int(max(0, seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"


def eval_progress(label, done, total, started_at):
    percentage = 100.0 * done / total if total else 100.0
    print(
        f"[PROGRESS] {label}: {done}/{total} ({percentage:.2f}%) "
        f"elapsed={eval_format_seconds(time.time() - started_at)}",
        flush=True,
    )


def stable_eval_seed(sample_id):
    digest = hashlib.sha256(str(sample_id).encode("utf-8")).digest()
    return (SEED + int.from_bytes(digest[:4], "big")) % (2**31 - 1)


def append_eval_jsonl(path, record):
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")
        handle.flush()
        os.fsync(handle.fileno())


def eval_record_key(sample_id, sample_index):
    return f"{sample_id}::sample-{int(sample_index):02d}"


def load_matching_eval_records(path):
    records = {}
    if not path.is_file():
        return records
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                print(f"[WARN] Ignoring truncated line {path}:{line_number}", flush=True)
                continue
            if record.get("run_fingerprint") == EVAL_RUN_FINGERPRINT:
                key = eval_record_key(record["sample_id"], record.get("sample_index", 0))
                records[key] = record
    return records


def split_eval_thinking(text):
    text = text.strip()
    if "</think>" in text:
        thinking, final_answer = text.rsplit("</think>", 1)
        thinking = re.sub(r"^\s*<think>\s*", "", thinking).strip()
        return thinking, final_answer.strip(), True
    return "", text, False


def extract_last_boxed(text):
    starts = [match.start() for match in re.finditer(r"\\boxed\s*\{", text)]
    for start in reversed(starts):
        brace_start = text.find("{", start)
        depth = 0
        for index in range(brace_start, len(text)):
            if text[index] == "{":
                depth += 1
            elif text[index] == "}":
                depth -= 1
                if depth == 0:
                    return text[brace_start + 1:index]
    return None


def extract_math_answer(text):
    boxed = extract_last_boxed(text)
    if boxed is not None:
        return boxed
    final_matches = re.findall(r"(?i)final\s+answer\s*(?:is|:|=)?\s*([^\n]+)", text)
    if final_matches:
        return final_matches[-1].strip()
    integers = re.findall(r"(?<![\w.])-?\d+(?![\w.])", text.replace(",", ""))
    return integers[-1] if integers else ""


def normalize_math_answer(value):
    value = str(value).strip().replace("$", "").replace(",", "")
    value = re.sub(r"\\(?:mathrm|text)\{([^{}]*)\}", r"\1", value)
    value = re.sub(r"\s+", "", value)
    if re.fullmatch(r"[+-]?\d+\.0+", value):
        value = str(int(float(value)))
    if re.fullmatch(r"[+-]?\d+", value):
        value = str(int(value))
    return value


def generate_eval_answer(problem, sample_key):
    prompt = (
        str(problem).strip()
        + "\n\nPlease reason step by step, and put your final answer within \\boxed{}."
    )
    messages = [{"role": "user", "content": prompt}]
    rendered = eval_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=bool(EVAL_ENABLE_THINKING),
    )
    encoded = eval_tokenizer(rendered, return_tensors="pt", add_special_tokens=False)
    input_ids = encoded["input_ids"].to("cuda:0")
    attention_mask = encoded["attention_mask"].to("cuda:0")
    input_tokens = int(input_ids.shape[1])
    available = EVAL_MAX_SEQ_LENGTH - input_tokens
    if available < 1:
        raise ValueError(f"Evaluation prompt has {input_tokens} tokens, exceeding context")
    effective_max_new_tokens = min(EVAL_MAX_NEW_TOKENS, available)

    sample_seed = stable_eval_seed(sample_key)
    random.seed(sample_seed)
    np.random.seed(sample_seed)
    torch.manual_seed(sample_seed)
    torch.cuda.manual_seed_all(sample_seed)
    started = time.time()
    with torch.inference_mode():
        generated = eval_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=effective_max_new_tokens,
            do_sample=True,
            temperature=EVAL_TEMPERATURE,
            top_p=EVAL_TOP_P,
            top_k=EVAL_TOP_K,
            min_p=EVAL_MIN_P,
            eos_token_id=eval_tokenizer.eos_token_id,
            pad_token_id=eval_tokenizer.pad_token_id,
            use_cache=True,
        )
    new_tokens = generated[0, input_tokens:]
    raw_generation = eval_tokenizer.decode(new_tokens, skip_special_tokens=True)
    thinking, final_answer, found_think_close = split_eval_thinking(raw_generation)
    return {
        "sample_seed": sample_seed,
        "raw_generation": raw_generation,
        "thinking": thinking,
        "final_answer": final_answer,
        "found_think_close": found_think_close,
        "input_tokens": input_tokens,
        "output_tokens": int(new_tokens.numel()),
        "effective_max_new_tokens": int(effective_max_new_tokens),
        "generation_seconds": time.time() - started,
    }


In [ ]:
def run_aime_evaluation(benchmark, dataset):
    output_path = raw_evaluation_dir / f"{benchmark}.jsonl"
    prior = load_matching_eval_records(output_path) if RESUME_EVALUATION else {}
    completed = {key for key, record in prior.items() if record.get("status") == "ok"}
    sample_ids = [str(row.get("id", index)) for index, row in enumerate(dataset)]
    expected_keys = {
        eval_record_key(sample_id, sample_index)
        for sample_id in sample_ids
        for sample_index in range(EVAL_SAMPLES_PER_PROMPT)
    }
    total_generations = len(expected_keys)
    if RESUME_EVALUATION and expected_keys.issubset(completed):
        print(
            f"[SKIP] client-math-fullft/{benchmark}/avg@{EVAL_SAMPLES_PER_PROMPT}: "
            f"{total_generations}/{total_generations} generations complete", flush=True
        )
        return prior

    started = time.time()
    done = len(expected_keys.intersection(completed))
    consecutive_errors = 0
    if completed:
        print(
            f"[RESUME] client-math-fullft/{benchmark}/avg@{EVAL_SAMPLES_PER_PROMPT}: "
            f"{done}/{total_generations} generations complete", flush=True
        )
    for index, example in enumerate(dataset):
        sample_id = str(example.get("id", index))
        for sample_index in range(EVAL_SAMPLES_PER_PROMPT):
            record_key = eval_record_key(sample_id, sample_index)
            if record_key in completed:
                continue
            base = {
                "run_fingerprint": EVAL_RUN_FINGERPRINT,
                "benchmark": benchmark,
                "sample_id": sample_id,
                "sample_index": sample_index,
                "samples_per_prompt": EVAL_SAMPLES_PER_PROMPT,
                "evaluation_id": record_key,
                "problem": str(example["problem"]),
                "gold_answer": str(example["answer"]),
            }
            try:
                generated = generate_eval_answer(
                    example["problem"], f"{benchmark}:{sample_id}:sample-{sample_index}"
                )
                predicted = extract_math_answer(generated["final_answer"])
                record = {
                    **base,
                    **generated,
                    "predicted_answer": predicted,
                    "correct": normalize_math_answer(predicted)
                    == normalize_math_answer(example["answer"]),
                    "status": "ok",
                }
                consecutive_errors = 0
            except Exception as exc:
                record = {**base, "status": "error", "error": repr(exc), "correct": False}
                consecutive_errors += 1
                print(f"[ERROR] {benchmark}:{sample_id}:sample-{sample_index}: {exc!r}", flush=True)
                torch.cuda.empty_cache()
            append_eval_jsonl(output_path, record)
            prior[record_key] = record
            done += 1
            eval_progress(
                f"client-math-fullft/{benchmark}/avg@{EVAL_SAMPLES_PER_PROMPT}",
                done, total_generations, started
            )
            if consecutive_errors >= MAX_CONSECUTIVE_EVAL_ERRORS:
                raise RuntimeError(
                    f"Aborting {benchmark} after {consecutive_errors} consecutive errors; "
                    f"last error: {record.get('error')}"
                )
    final_records = load_matching_eval_records(output_path)
    successful = {key for key, record in final_records.items() if record.get("status") == "ok"}
    missing = expected_keys - successful
    if missing:
        raise RuntimeError(
            f"{benchmark} Avg@{EVAL_SAMPLES_PER_PROMPT} is incomplete: "
            f"{len(missing)} generations missing or failed"
        )
    return final_records


aime_records = {
    benchmark: run_aime_evaluation(benchmark, dataset)
    for benchmark, dataset in aime_datasets.items()
}


## AIME metrics and standalone bar chart


In [ ]:
metric_rows = []
for benchmark, dataset in aime_datasets.items():
    records_by_id = aime_records[benchmark]
    selected_ids = [str(row.get("id", index)) for index, row in enumerate(dataset)]
    selected_records = [
        records_by_id[eval_record_key(sample_id, sample_index)]
        for sample_id in selected_ids
        for sample_index in range(EVAL_SAMPLES_PER_PROMPT)
    ]
    correct = sum(bool(record.get("correct")) for record in selected_records)
    total = len(selected_records)
    per_problem = []
    for sample_id in selected_ids:
        problem_records = [
            records_by_id[eval_record_key(sample_id, sample_index)]
            for sample_index in range(EVAL_SAMPLES_PER_PROMPT)
        ]
        problem_correct = sum(bool(record.get("correct")) for record in problem_records)
        per_problem.append({
            "sample_id": sample_id,
            "correct_samples": problem_correct,
            "total_samples": EVAL_SAMPLES_PER_PROMPT,
            "average_accuracy": problem_correct / EVAL_SAMPLES_PER_PROMPT,
        })
    metric_rows.append({
        "benchmark": benchmark.upper(),
        "metric": f"avg_at_{EVAL_SAMPLES_PER_PROMPT}",
        "correct_generations": correct,
        "total_generations": total,
        "problems": len(selected_ids),
        "samples_per_prompt": EVAL_SAMPLES_PER_PROMPT,
        "score": correct / total if total else None,
        "score_percent": 100.0 * correct / total if total else None,
        "per_problem": per_problem,
    })

aime_metrics = pd.DataFrame(metric_rows)
metrics_csv = evaluation_dir / "aime25_aime26_metrics.csv"
metrics_json = evaluation_dir / "aime25_aime26_metrics.json"
bar_chart_path = evaluation_dir / "aime25_aime26_avg_at_12.png"
aime_metrics.to_csv(metrics_csv, index=False, encoding="utf-8-sig")
metrics_json.write_text(
    json.dumps(
        {
            "model": str(run_dir),
            "run_fingerprint": EVAL_RUN_FINGERPRINT,
            "protocol": eval_fingerprint_payload,
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "metrics": metric_rows,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

labels = aime_metrics["benchmark"].tolist()
values = aime_metrics["score_percent"].astype(float).tolist()
figure, axis = plt.subplots(figsize=(7.5, 5.5))
bars = axis.bar(labels, values, color=["#4C78A8", "#72B7B2"], width=0.58,
                edgecolor="#333333", linewidth=0.8)
for bar, value, correct, total in zip(
    bars, values, aime_metrics["correct_generations"], aime_metrics["total_generations"]
):
    axis.text(
        bar.get_x() + bar.get_width() / 2,
        value + 1.2,
        f"{value:.2f}%\n({correct}/{total} generations)",
        ha="center",
        va="bottom",
        fontsize=10,
    )
axis.set_ylabel(f"Avg@{EVAL_SAMPLES_PER_PROMPT} exact-match accuracy (%)")
axis.set_ylim(0, 108)
axis.set_title(f"Full-parameter OPSD Math Student - AIME Avg@{EVAL_SAMPLES_PER_PROMPT}")
axis.grid(axis="y", linestyle="--", alpha=0.3)
figure.tight_layout()
figure.savefig(bar_chart_path, dpi=200, bbox_inches="tight")
plt.show()

display(aime_metrics)
print(f"Metrics JSON: {metrics_json}", flush=True)
print(f"Metrics CSV:  {metrics_csv}", flush=True)
print(f"Bar chart:    {bar_chart_path}", flush=True)

del eval_model
gc.collect()
torch.cuda.empty_cache()


## Interpretation boundary

The AIME25/AIME26 bars report **Avg@12**: each problem receives 12 independently seeded
generations, and the final score is the mean exact-match accuracy over all generations.
This matches the paper's aggregation metric but not its full decoding protocol: this notebook
retains the project settings (thinking disabled, temperature 0.6, and 1024 new tokens), whereas
the paper uses thinking enabled, temperature 1.0, and up to 38912 new tokens.

For a 30-problem benchmark, Avg@12 uses 360 generations and has a nominal resolution of
1/360 = 0.278 percentage points. It is not pass@12: a problem answered correctly once in 12
contributes 1/12 rather than being counted as fully solved.
